# Style-Matching Level Generation via KDE-Sampled GA

Generate novel Mummy Maze levels that match the style of the original 10,100 levels.

**Approach**:
1. Classify original levels into **bins** by entity composition (has_mummy2, has_scorpion, trap_count, gate_relevance) and grid size
2. Fit a KDE per bin on continuous features (BFS moves, n_states, log win prob, wall count)
3. For each new level: sample a target from the bin's KDE, run a short GA to match it
4. The GA generates levels with the correct entity composition and matching structural style

In [ ]:
import random

import numpy as np
import mummymaze_rust
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.stats import gaussian_kde
from IPython.display import HTML

MAZE_DIR = Path("../mazes")

# ============================================================
# CONFIGURATION
# ============================================================

# Grid sizes to include
GRID_SIZES = [8]

# Entity composition bins to generate. Set to None to use all bins
# that have enough originals (>= MIN_BIN_SIZE).
# Format: (has_mummy2, has_scorpion, trap_count, trap_bin, gate_bin)
# trap_bin: "none" = no traps, "decorative" = traps don't affect BFS, "functional" = traps matter
# gate_bin: "none" = no gate
#           "decorative" = gate doesn't affect BFS
#           "passage" = gate needed as open passage, but toggling doesn't matter
#           "functional" = toggling mechanic changes the optimal solution
ACTIVE_BINS = None  # None = auto-select all bins with enough data

# Minimum number of original levels in a bin to fit a KDE
MIN_BIN_SIZE = 20

# --- GA parameters ---
N_LEVELS_PER_BIN = 5    # how many levels to generate per active bin
GA_POP_SIZE = 64        # population size for each short GA run
GA_GENERATIONS = 100    # generations per GA run
GA_TOURNAMENT_K = 2     # tournament selection pressure (higher = greedier)
GA_ELITE_FRAC = 0.10    # fraction of population carried over unchanged
GA_MUTATION_RATE = 0.8  # probability of mutation vs crossover
GA_CROSSOVER_MODES = ["swap_entities", "wall_patch", "region"]

# Mutation weights (w_add_entity and w_remove_entity are always 0 to preserve bin)
GA_W_WALL = 5.0
GA_W_MOVE_ENTITY = 3.0
GA_W_MOVE_PLAYER = 2.0
GA_W_MOVE_EXIT = 1.0
GA_EXTRA_WALL_PROB = 0.3

SEED = 42

## 1. Parse levels, classify into bins, compute features

In [ ]:
def wall_count(level):
    """Count interior walls (excluding border) from edge arrays."""
    d = level.to_dict()
    gs = d["grid_size"]
    h = d["h_walls"]
    v = d["v_walls"]
    h_interior = sum(h[i] for i in range(gs, gs * gs))
    v_interior = 0
    for r in range(gs):
        for c in range(1, gs):
            v_interior += v[r * (gs + 1) + c]
    return h_interior + v_interior


def level_features(eval_result, level):
    """Extract continuous feature vector."""
    return np.array([
        eval_result["bfs_moves"],
        eval_result["n_states"],
        eval_result["log_win_prob"],
        wall_count(level),
    ], dtype=np.float64)


FEATURE_NAMES = ["BFS moves", "N states", "Log win prob", "Wall count"]


def classify_gate(level, bfs_with):
    """Classify gate as none/decorative/passage/functional."""
    d = level.to_dict()
    if not d.get("gate"):
        return "none"

    # Without gate (open passage)
    d_no_gate = {**d, "gate": None, "key": None}
    lev_no_gate = mummymaze_rust.Level.from_dict(d_no_gate)
    bfs_without = mummymaze_rust.solve(lev_no_gate)

    if bfs_without is not None and bfs_without == bfs_with:
        return "decorative"

    # With permanent wall instead of gate
    gs = d["grid_size"]
    gr, gc = d["gate"]
    v_walls = list(d["v_walls"])
    v_walls[gr * (gs + 1) + gc + 1] = True
    d_wall = {**d, "gate": None, "key": None, "v_walls": v_walls}
    lev_wall = mummymaze_rust.Level.from_dict(d_wall)
    bfs_wall = mummymaze_rust.solve(lev_wall)

    if bfs_wall is not None and bfs_wall == bfs_with:
        return "passage"

    return "functional"


def classify_traps(level, bfs_with):
    """Classify traps as none/decorative/functional."""
    d = level.to_dict()
    traps = d.get("traps", [])
    if not traps:
        return "none"

    d_no_traps = {**d, "traps": []}
    lev_no_traps = mummymaze_rust.Level.from_dict(d_no_traps)
    bfs_without = mummymaze_rust.solve(lev_no_traps)

    if bfs_without is not None and bfs_without == bfs_with:
        return "decorative"
    return "functional"


def level_bin(level, bfs_moves):
    """Compute the entity-composition bin key for a level."""
    d = level.to_dict()
    has_mummy2 = d["mummy2"] is not None
    has_scorpion = d["scorpion"] is not None
    trap_count = len(d.get("traps", []))
    gate_bin = classify_gate(level, bfs_moves)
    trap_bin = classify_traps(level, bfs_moves)
    return (has_mummy2, has_scorpion, trap_count, trap_bin, gate_bin)


def bin_label(b):
    """Human-readable label for a bin key."""
    m2, sc, tc, tb, gb = b
    parts = ["2M" if m2 else "1M"]
    if sc:
        parts.append("S")
    if tc > 0:
        parts.append(f"{tc}T:{tb}")
    if gb != "none":
        parts.append(f"G:{gb}")
    return " ".join(parts)


# Parse all levels
print("Parsing levels...")
all_levels_flat = {}
for gs in GRID_SIZES:
    all_levels_flat[gs] = []

for dat_file in sorted(MAZE_DIR.glob("B-*.dat")):
    try:
        levels = mummymaze_rust.parse_file(str(dat_file))
    except Exception:
        continue
    for lev in levels:
        d = lev.to_dict()
        gs = d["grid_size"]
        if gs in all_levels_flat:
            all_levels_flat[gs].append(lev)

for gs in GRID_SIZES:
    print(f"  gs={gs}: {len(all_levels_flat[gs])} levels")

# Evaluate and classify into bins
print("\nEvaluating and classifying...")

binned_data = {}
original_features = {}

for gs in GRID_SIZES:
    levels = all_levels_flat[gs]
    results = mummymaze_rust.ga_evaluate_batch(levels)

    gs_feats = []
    for r in results:
        lev = r["level"]
        feat = level_features(r, lev)
        bfs = r["bfs_moves"]
        b = level_bin(lev, bfs)

        key = (gs, b)
        if key not in binned_data:
            binned_data[key] = []
        binned_data[key].append((lev, feat))
        gs_feats.append(feat)

    original_features[gs] = np.array(gs_feats)
    print(f"  gs={gs}: {len(gs_feats)} solvable")

# Show bin distribution
print("\nBin distribution:")
for (gs, b), items in sorted(binned_data.items()):
    print(f"  gs={gs} {bin_label(b):30s} ({b}): {len(items)} levels")

### Feature distributions + skewness analysis

In [ ]:
from scipy.stats import skew, kurtosis

for gs in GRID_SIZES:
    feats = original_features[gs]
    print(f"\n=== gs={gs} ({feats.shape[0]} levels) ===")
    fig, axes = plt.subplots(2, 4, figsize=(18, 7))

    for i, name in enumerate(FEATURE_NAMES):
        col = feats[:, i]
        sk = skew(col)
        ku = kurtosis(col)

        # Raw distribution
        axes[0, i].hist(col, bins=40, color="tab:blue", alpha=0.7)
        axes[0, i].set_title(f"{name}\nskew={sk:.2f} kurt={ku:.2f}")
        axes[0, i].set_ylabel("Count")

        # Log-transformed (for positive features)
        if col.min() > 0:
            log_col = np.log(col)
            log_sk = skew(log_col)
            axes[1, i].hist(log_col, bins=40, color="tab:orange", alpha=0.7)
            axes[1, i].set_title(f"log({name})\nskew={log_sk:.2f}")
        else:
            # log1p for features that can be 0 or negative
            shifted = col - col.min() + 1
            log_col = np.log(shifted)
            log_sk = skew(log_col)
            axes[1, i].hist(log_col, bins=40, color="tab:orange", alpha=0.7)
            axes[1, i].set_title(f"log({name} - min + 1)\nskew={log_sk:.2f}")
        axes[1, i].set_ylabel("Count")

    axes[0, 0].set_ylabel("Raw", fontsize=12, fontweight="bold")
    axes[1, 0].set_ylabel("Log-transformed", fontsize=12, fontweight="bold")
    fig.suptitle(f"Feature Distributions: Raw vs Log (gs={gs})", fontsize=14)
    plt.tight_layout()
    plt.show()

## 2. Fit per-bin KDEs + select active bins

In [ ]:
# Fit a KDE for each bin that has enough data.
# N states (index 1) is log-transformed before fitting to reduce skew.
# Features are normalized, then PCA handles remaining degeneracies.

from sklearn.decomposition import PCA

LOG_DIMS = [1]  # which feature indices to log-transform (just N states)


def transform_features(feats):
    """Log-transform skewed dimensions for KDE fitting."""
    out = feats.copy()
    for d in LOG_DIMS:
        out[:, d] = np.log1p(np.maximum(out[:, d], 0))
    return out


def untransform_features(feats):
    """Invert log transform."""
    out = feats.copy()
    for d in LOG_DIMS:
        out[:, d] = np.expm1(out[:, d])
    return out


class BinKDE:
    """KDE wrapper: log-transforms N states, normalizes, PCA for degeneracies."""
    def __init__(self, feats):
        self.n_features = feats.shape[1]
        self.n_samples = feats.shape[0]

        transformed = transform_features(feats)
        self.t_mean = transformed.mean(axis=0)
        self.t_std = transformed.std(axis=0)
        self.t_std[self.t_std < 1e-8] = 1.0
        normed = (transformed - self.t_mean) / self.t_std

        pca = PCA()
        pca.fit(normed)
        explained = pca.explained_variance_ratio_
        n_components = max(1, int((explained > 0.001).sum()))
        self.pca = PCA(n_components=n_components)
        self.pca_data = self.pca.fit_transform(normed)
        self.n_effective = n_components

        if n_components >= 1 and self.n_samples >= 3:
            self.kde = gaussian_kde(self.pca_data.T)
        else:
            self.kde = None

    def resample(self, n, seed=None):
        """Sample n target vectors in original feature space."""
        if self.kde is not None:
            samples_pca = self.kde.resample(n, seed=seed).T
            normed = self.pca.inverse_transform(samples_pca)
            transformed = normed * self.t_std + self.t_mean
            return untransform_features(transformed)
        else:
            return np.tile(untransform_features(
                self.t_mean.reshape(1, -1)
            )[0], (n, 1))


bin_kdes = {}
bin_scales = {}
bin_features = {}

for key, items in sorted(binned_data.items()):
    gs, b = key
    if len(items) < MIN_BIN_SIZE:
        continue
    feats = np.array([f for _, f in items])
    bin_features[key] = feats
    std = feats.std(axis=0)
    std[std < 1e-8] = 1.0
    bin_scales[key] = std
    bin_kdes[key] = BinKDE(feats)

# Determine active bins
if ACTIVE_BINS is not None:
    active_keys = [(gs, b) for gs in GRID_SIZES for b in ACTIVE_BINS if (gs, b) in bin_kdes]
else:
    active_keys = sorted(bin_kdes.keys())

print(f"Bins with KDE (>= {MIN_BIN_SIZE} levels): {len(bin_kdes)}")
print(f"Active bins for generation: {len(active_keys)}\n")
for key in active_keys:
    gs, b = key
    n = len(binned_data[key])
    bk = bin_kdes[key]
    print(f"  gs={gs} {bin_label(b):25s}: {n:4d} levels, {bk.n_effective}D KDE (from {bk.n_features}D)")

In [ ]:
# Visualize bin sizes as a bar chart
fig, ax = plt.subplots(figsize=(14, 5))
labels = [f"gs={gs}\n{bin_label(b)}" for gs, b in sorted(binned_data.keys())]
counts = [len(items) for _, items in sorted(binned_data.items())]
colors = ["tab:green" if k in bin_kdes else "tab:gray" for k in sorted(binned_data.keys())]
ax.bar(range(len(labels)), counts, color=colors)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=7)
ax.set_ylabel("Level count")
ax.set_title(f"Entity Composition Bins (green = KDE fitted, >= {MIN_BIN_SIZE} levels)")
ax.axhline(MIN_BIN_SIZE, color="red", linestyle="--", alpha=0.5, label=f"min={MIN_BIN_SIZE}")
ax.legend()
plt.tight_layout()
plt.show()

## 3. KDE-sampled GA with entity-composition constraints

In [ ]:
def random_level_for_bin(grid_size, entity_bin, rng=random):
    """Generate a random level matching the given entity-composition bin."""
    has_mummy2, has_scorpion, trap_count, _trap_bin, gate_bin = entity_bin
    n = grid_size
    flip = rng.random() < 0.5

    wall_prob = rng.uniform(0.15, 0.45)
    h_walls = []
    for r in range(n + 1):
        for c in range(n):
            h_walls.append(True if (r == 0 or r == n) else rng.random() < wall_prob)
    v_walls = []
    for r in range(n):
        for c in range(n + 1):
            v_walls.append(True if (c == 0 or c == n) else rng.random() < wall_prob)

    exit_side = rng.choice(["N", "S", "E", "W"])
    exit_pos = rng.randint(0, n - 1)

    occupied = set()
    def rand_pos():
        for _ in range(100):
            r, c = rng.randint(0, n - 1), rng.randint(0, n - 1)
            if (r, c) not in occupied:
                occupied.add((r, c))
                return (r, c)
        return (0, 0)

    player = rand_pos()
    mummy1 = rand_pos()
    mummy2 = rand_pos() if has_mummy2 else None
    scorpion = rand_pos() if has_scorpion else None
    traps = [rand_pos() for _ in range(trap_count)]

    # Gate: col must be < n-1 (east edge must be interior)
    gate = None
    key = None
    if gate_bin != "none":
        gr = rng.randint(0, n - 1)
        gc = rng.randint(0, n - 2)  # col < n-1
        gate = (gr, gc)
        # Clear east wall at gate cell so gate is visible
        v_walls[gr * (n + 1) + gc + 1] = False
        key = rand_pos()

    return mummymaze_rust.Level.from_edges(
        grid_size, flip, h_walls, v_walls,
        exit_side, exit_pos, player, mummy1,
        mummy2, scorpion, traps, gate, key,
    )


def target_fitness(feat, target, scales):
    """Negative normalized distance to a specific target vector."""
    return -np.sqrt(((feat - target) / scales) ** 2).sum()


def generate_one_level(grid_size, entity_bin, target, scales,
                       pop_size=GA_POP_SIZE, generations=GA_GENERATIONS,
                       tournament_k=GA_TOURNAMENT_K, elite_frac=GA_ELITE_FRAC,
                       mutation_rate=GA_MUTATION_RATE, crossover_modes=GA_CROSSOVER_MODES,
                       w_wall=GA_W_WALL, w_move_entity=GA_W_MOVE_ENTITY,
                       w_move_player=GA_W_MOVE_PLAYER, w_move_exit=GA_W_MOVE_EXIT,
                       extra_wall_prob=GA_EXTRA_WALL_PROB, rng=random):
    """Run a short GA to produce a level matching a target feature vector within a bin."""
    population = []
    for _ in range(20):
        batch = [random_level_for_bin(grid_size, entity_bin, rng) for _ in range(pop_size * 3)]
        results = mummymaze_rust.ga_evaluate_batch(batch)
        for r in results:
            lev = r["level"]
            feat = level_features(r, lev)
            fit = target_fitness(feat, target, scales)
            population.append((lev, feat, fit))
            if len(population) >= pop_size:
                break
        if len(population) >= pop_size:
            break

    if not population:
        return None

    population.sort(key=lambda x: x[2], reverse=True)

    for gen in range(generations):
        n_elite = max(1, int(len(population) * elite_frac))
        next_gen = list(population[:n_elite])

        offspring_levels = []
        while len(offspring_levels) < pop_size - n_elite:
            candidates = rng.sample(population, min(tournament_k, len(population)))
            parent = max(candidates, key=lambda x: x[2])
            if rng.random() < mutation_rate:
                child = mummymaze_rust.mutate(
                    parent[0], rng.randint(0, 2**63),
                    w_wall=w_wall, w_move_entity=w_move_entity,
                    w_move_player=w_move_player, w_move_exit=w_move_exit,
                    w_add_entity=0.0, w_remove_entity=0.0,
                    extra_wall_prob=extra_wall_prob,
                )
            else:
                candidates2 = rng.sample(population, min(tournament_k, len(population)))
                parent2 = max(candidates2, key=lambda x: x[2])
                mode = rng.choice(crossover_modes)
                child = mummymaze_rust.ga_crossover(
                    parent[0], parent2[0], mode, rng.randint(0, 2**63),
                )
            offspring_levels.append(child)

        results = mummymaze_rust.ga_evaluate_batch(offspring_levels)
        for r in results:
            lev = r["level"]
            feat = level_features(r, lev)
            fit = target_fitness(feat, target, scales)
            next_gen.append((lev, feat, fit))

        next_gen.sort(key=lambda x: x[2], reverse=True)
        population = next_gen[:pop_size]

    return population[0]


def generate_levels_binned(active_keys, n_per_bin=N_LEVELS_PER_BIN, seed=SEED, **ga_kwargs):
    """Generate levels across all active bins."""
    rng = random.Random(seed)
    np_rng = np.random.RandomState(seed)

    all_generated = {}

    for key in active_keys:
        gs, entity_bin = key
        bkde = bin_kdes[key]
        scales = bin_scales[key]

        print(f"\ngs={gs} {bin_label(entity_bin)} — generating {n_per_bin} levels...")
        generated = []

        for i in range(n_per_bin):
            target = bkde.resample(1, seed=np_rng)[0]
            result = generate_one_level(gs, entity_bin, target, scales, rng=rng, **ga_kwargs)
            if result is not None:
                lev, feat, fit = result
                generated.append((lev, feat, fit, target))

        all_generated[key] = generated
        if generated:
            fits = [g[2] for g in generated]
            print(f"  {len(generated)}/{n_per_bin} generated, "
                  f"fitness: best={max(fits):.3f} avg={np.mean(fits):.3f}")
        else:
            print(f"  0/{n_per_bin} generated (no solvable levels found)")

    return all_generated

In [ ]:
# Generate levels for all active bins
all_generated = generate_levels_binned(
    active_keys,
    n_per_bin=N_LEVELS_PER_BIN,
    pop_size=GA_POP_SIZE,
    generations=GA_GENERATIONS,
    seed=SEED,
)

total = sum(len(v) for v in all_generated.values())
print(f"\nTotal generated: {total} levels across {len(all_generated)} bins")

## 4. Maze renderer + BFS solution animation

In [ ]:
ENTITY_RADIUS = 0.3
TRAP_SIZE = 0.25

def draw_maze(ax, level_dict, frame=None, title=None):
    """Draw a maze state on a matplotlib axes.

    Args:
        ax: matplotlib axes
        level_dict: from level.to_dict()
        frame: optional state dict from replay_actions (entity positions for this step)
        title: optional title string
    """
    ax.clear()
    gs = level_dict["grid_size"]

    # Grid coordinates: column = x, row = y (inverted so row 0 is at top)
    ax.set_xlim(-0.5, gs - 0.5)
    ax.set_ylim(gs - 0.5, -0.5)  # inverted y
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])

    # Background
    ax.set_facecolor("#f5f0e0")

    # Draw grid cells
    for r in range(gs):
        for c in range(gs):
            rect = plt.Rectangle((c - 0.5, r - 0.5), 1, 1,
                                  linewidth=0.5, edgecolor="#d0c8b0",
                                  facecolor="#f5f0e0")
            ax.add_patch(rect)

    # Draw walls from h_walls / v_walls
    h_walls = level_dict["h_walls"]  # (gs+1) rows, gs cols
    v_walls = level_dict["v_walls"]  # gs rows, (gs+1) cols
    wall_color = "#2a1a0a"
    wall_lw = 3.0

    for r in range(gs + 1):
        for c in range(gs):
            if h_walls[r * gs + c]:
                ax.plot([c - 0.5, c + 0.5], [r - 0.5, r - 0.5],
                        color=wall_color, linewidth=wall_lw, solid_capstyle="round")

    for r in range(gs):
        for c in range(gs + 1):
            if v_walls[r * (gs + 1) + c]:
                ax.plot([c - 0.5, c - 0.5], [r - 0.5, r + 0.5],
                        color=wall_color, linewidth=wall_lw, solid_capstyle="round")

    # Draw exit opening
    exit_side = level_dict["exit_side"]
    exit_pos = level_dict["exit_pos"]
    exit_color = "#4CAF50"
    exit_lw = 4.0
    if exit_side == "N":
        ax.plot([exit_pos - 0.4, exit_pos + 0.4], [-0.5, -0.5],
                color=exit_color, linewidth=exit_lw, solid_capstyle="round")
    elif exit_side == "S":
        ax.plot([exit_pos - 0.4, exit_pos + 0.4], [gs - 0.5, gs - 0.5],
                color=exit_color, linewidth=exit_lw, solid_capstyle="round")
    elif exit_side == "W":
        ax.plot([-0.5, -0.5], [exit_pos - 0.4, exit_pos + 0.4],
                color=exit_color, linewidth=exit_lw, solid_capstyle="round")
    elif exit_side == "E":
        ax.plot([gs - 0.5, gs - 0.5], [exit_pos - 0.4, exit_pos + 0.4],
                color=exit_color, linewidth=exit_lw, solid_capstyle="round")

    # Entity positions — use frame if provided, otherwise use initial from level_dict
    if frame is not None:
        pr, pc = frame["player"]
        m1r, m1c, m1alive = frame["mummy1"]
        m2r, m2c, m2alive = frame["mummy2"]
        sr, sc, salive = frame["scorpion"]
        gate_open = frame["gate_open"]
    else:
        pr, pc = level_dict["player"]
        m1r, m1c = level_dict["mummy1"]
        m1alive = True
        m2 = level_dict["mummy2"]
        m2r, m2c, m2alive = (m2[0], m2[1], True) if m2 else (0, 0, False)
        scorp = level_dict["scorpion"]
        sr, sc, salive = (scorp[0], scorp[1], True) if scorp else (0, 0, False)
        gate_open = False

    # Traps (X marks)
    for tr, tc in level_dict.get("traps", []):
        s = TRAP_SIZE
        ax.plot([tc - s, tc + s], [tr - s, tr + s], color="#8B0000", linewidth=2.5)
        ax.plot([tc - s, tc + s], [tr + s, tr - s], color="#8B0000", linewidth=2.5)

    # Key
    if level_dict.get("key"):
        kr, kc = level_dict["key"]
        ax.plot(kc, kr, marker="P", markersize=12, color="#FFD700",
                markeredgecolor="#B8860B", markeredgewidth=1.5, zorder=5)

    # Gate
    if level_dict.get("gate"):
        gr, gc = level_dict["gate"]
        blocking = gate_open
        gate_color = "#8B0000" if blocking else "#90EE90"
        ax.plot([gc + 0.5, gc + 0.5], [gr - 0.35, gr + 0.35],
                color=gate_color, linewidth=4, solid_capstyle="round", zorder=6)

    # Mummy 1
    mummy_color = "#D32F2F" if level_dict["flip"] else "#F5F5DC"
    mummy_edge = "#8B0000" if level_dict["flip"] else "#8B8B00"
    if m1alive:
        ax.add_patch(plt.Circle((m1c, m1r), ENTITY_RADIUS, color=mummy_color,
                                ec=mummy_edge, linewidth=2, zorder=8))
        ax.text(m1c, m1r, "M", ha="center", va="center", fontsize=9,
                fontweight="bold", color=mummy_edge, zorder=9)

    # Mummy 2
    if m2alive:
        ax.add_patch(plt.Circle((m2c, m2r), ENTITY_RADIUS, color=mummy_color,
                                ec=mummy_edge, linewidth=2, zorder=8))
        ax.text(m2c, m2r, "M", ha="center", va="center", fontsize=9,
                fontweight="bold", color=mummy_edge, zorder=9)

    # Scorpion
    if salive:
        ax.add_patch(plt.Circle((sc, sr), ENTITY_RADIUS, color="#FF8C00",
                                ec="#8B4500", linewidth=2, zorder=8))
        ax.text(sc, sr, "S", ha="center", va="center", fontsize=9,
                fontweight="bold", color="#8B4500", zorder=9)

    # Player
    result = frame["result"] if frame else "ok"
    player_color = "#4FC3F7" if result == "ok" else ("#4CAF50" if result == "win" else "#EF5350")
    ax.add_patch(plt.Circle((pc, pr), ENTITY_RADIUS, color=player_color,
                            ec="#01579B", linewidth=2, zorder=10))
    ax.text(pc, pr, "P", ha="center", va="center", fontsize=9,
            fontweight="bold", color="#01579B", zorder=11)

    if title:
        ax.set_title(title, fontsize=11)


def animate_solution(level, interval=500, figsize=(5, 5)):
    """Animate the BFS solution of a level. Returns a JS-based animation for notebook display."""
    actions = mummymaze_rust.solve_actions(level)
    if actions is None:
        print("Level is unsolvable!")
        return None

    frames = mummymaze_rust.replay_actions(level, actions)
    level_dict = level.to_dict()
    action_names = ["N", "S", "E", "W", "Wait"]

    fig, ax = plt.subplots(1, 1, figsize=figsize)

    def update(i):
        if i == 0:
            title = f"Step 0/{len(actions)} (initial)"
        else:
            title = f"Step {i}/{len(actions)} (action: {action_names[actions[i-1]]})"
        draw_maze(ax, level_dict, frames[i], title=title)

    anim = FuncAnimation(fig, update, frames=len(frames), interval=interval, repeat=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())


# Test: draw a random original level
test_lev = all_levels_flat[8][42]
fig, ax = plt.subplots(figsize=(5, 5))
draw_maze(ax, test_lev.to_dict(), title="Original level (gs=8, #42)")
plt.tight_layout()
plt.show()

In [ ]:
# Animate the BFS solution of that original level
animate_solution(all_levels_flat[8][42], interval=600)

## 5. Compare: generated vs original levels (per bin)

In [ ]:
# Show 3 generated levels per bin (first 6 bins), with nearest original
show_keys = [k for k in active_keys if len(all_generated.get(k, [])) >= 3][:6]
n_cols = 3

fig, axes = plt.subplots(2 * len(show_keys), n_cols, figsize=(6 * n_cols, 6 * 2 * len(show_keys)))
if len(show_keys) == 1:
    axes = axes.reshape(2, n_cols)

for row_idx, key in enumerate(show_keys):
    gs, entity_bin = key
    generated = all_generated[key]
    orig_items = binned_data[key]
    orig_feats_bin = bin_features[key]
    scales = bin_scales[key]

    for col in range(min(n_cols, len(generated))):
        lev, feat, fit, target = generated[col]

        # Find nearest original in same bin
        normalized = (feat - orig_feats_bin) / scales
        dists = np.linalg.norm(normalized, axis=1)
        nn_idx = dists.argmin()
        nn_dist = dists[nn_idx]

        ax_gen = axes[row_idx * 2, col]
        ax_orig = axes[row_idx * 2 + 1, col]

        draw_maze(ax_gen, lev.to_dict(),
                  title=f"Generated (gs={gs} {bin_label(entity_bin)})\nfit={fit:.3f}")
        draw_maze(ax_orig, orig_items[nn_idx][0].to_dict(),
                  title=f"Nearest original\nnn_dist={nn_dist:.3f}")

    axes[row_idx * 2, 0].set_ylabel("Generated", fontsize=11, fontweight="bold")
    axes[row_idx * 2 + 1, 0].set_ylabel("Original", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# Animate a generated level from the first active bin
first_key = show_keys[0] if show_keys else active_keys[0]
animate_solution(all_generated[first_key][0][0], interval=600)

## 6. Feature distribution comparison (per grid size, all bins combined)

In [ ]:
# Aggregate generated features per grid size
for gs in GRID_SIZES:
    gen_feats_gs = []
    for key, items in all_generated.items():
        if key[0] == gs:
            gen_feats_gs.extend([f for _, f, _, _ in items])
    if not gen_feats_gs:
        continue
    gen_feats_gs = np.array(gen_feats_gs)
    orig_feats_gs = original_features[gs]

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    for i, (ax, name) in enumerate(zip(axes, FEATURE_NAMES)):
        # Compute shared bin edges from combined data range
        combined = np.concatenate([orig_feats_gs[:, i], gen_feats_gs[:, i]])
        bins = np.linspace(combined.min(), combined.max(), 31)
        ax.hist(orig_feats_gs[:, i], bins=bins, alpha=0.5, label="Original", density=True, color="tab:blue")
        ax.hist(gen_feats_gs[:, i], bins=bins, alpha=0.6, label="Generated", density=True, color="tab:orange")
        ax.set_xlabel(name)
        ax.set_ylabel("Density")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    fig.suptitle(f"Feature Distributions: Original vs Generated (gs={gs}, {len(gen_feats_gs)} levels)", fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# 2D scatter: generated levels overlaid on original KDE density (gs=8)
gs = 8
gen_feats_8 = []
target_feats_8 = []
for key, items in all_generated.items():
    if key[0] == gs:
        gen_feats_8.extend([f for _, f, _, _ in items])
        target_feats_8.extend([t for _, _, _, t in items])

if gen_feats_8:
    gen_feats_8 = np.array(gen_feats_8)
    target_feats_8 = np.array(target_feats_8)
    orig_feats_8 = original_features[gs]

    pairs = [(0, 1), (0, 2), (0, 3), (2, 3)]
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    for idx, (i, j) in enumerate(pairs):
        ax = axes[idx]
        of = orig_feats_8

        data_2d = of[:, [i, j]].T
        kde_2d = gaussian_kde(data_2d)
        xi = np.linspace(of[:, i].min() - of[:, i].std() * 0.5,
                         of[:, i].max() + of[:, i].std() * 0.5, 60)
        yi = np.linspace(of[:, j].min() - of[:, j].std() * 0.5,
                         of[:, j].max() + of[:, j].std() * 0.5, 60)
        Xi, Yi = np.meshgrid(xi, yi)
        Zi = kde_2d(np.vstack([Xi.ravel(), Yi.ravel()])).reshape(Xi.shape)
        ax.contourf(Xi, Yi, Zi, levels=15, cmap="Blues", alpha=0.6)

        ax.scatter(of[:, i], of[:, j], s=2, alpha=0.1, color="gray", label="Original")
        ax.scatter(target_feats_8[:, i], target_feats_8[:, j], s=40, marker="x",
                   color="red", alpha=0.7, linewidths=1.5, label="KDE target")
        ax.scatter(gen_feats_8[:, i], gen_feats_8[:, j], s=30, color="tab:orange",
                   edgecolors="black", linewidths=0.5, alpha=0.8, label="Generated")

        ax.set_xlabel(FEATURE_NAMES[i])
        ax.set_ylabel(FEATURE_NAMES[j])
        if idx == 0:
            ax.legend(fontsize=7, loc="upper right")

    fig.suptitle("Generated Levels in Feature Space (gs=8): KDE density + targets + results", fontsize=13)
    plt.tight_layout()
    plt.show()